# Stage 5 - adversarial training under two threat models (decisive, averaged)

Both datasets are evaluated the SAME way, under BOTH threat models, and averaged over repeats.

- **transfer**: attack crafted from the *baseline* CNN, fed to every model (a real attacker has no access to the defended model).
- **white-box**: attack crafted against the model it hits (worst case; the Random Forest has no gradients, so white-box does not apply to it).

The metric is robust-support macro-F1 (classes with >=2 test frames, computed the same way for both datasets). CICIoV single runs are unstable on tiny data, so the averaged table below is the one to read; the first cell just shows one run in detail.

In [ ]:
import sys
from pathlib import Path
try:
    import adversec
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().parent)); import adversec
import numpy as np, pandas as pd, io, contextlib
from adversec import config
import torch, joblib
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from adversec.models import CNN1D, train_cnn, build_random_forest
from adversec.experiments import attack as atk
from adversec.experiments.defense import adversarial_train_cnn
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', DEVICE)
DATASETS = ['ciciov2024', 'road']
def mf1(y, p, labels=None): return f1_score(y, p, labels=labels, average='macro', zero_division=0)

# load arrays once; robust-support = classes with >=2 test frames (computed, equal for both datasets)
data = {}
for name in DATASETS:
    a = np.load(config.PROCESSED_DIR / f'{name}_stage2_arrays.npz')
    classes = list(joblib.load(config.PROCESSED_DIR / f'{name}_label_encoder.joblib').classes_)
    cfg = config.load_dataset_config(name)
    counts = pd.Series(a['y_test']).value_counts()
    rs = [i for i in range(len(classes)) if counts.get(i, 0) >= 2]
    data[name] = dict(Xtr=a['X_train'], ytr=a['y_train'], Xte=a['X_test'].astype(np.float32),
                      yte=a['y_test'], classes=classes, cfg=cfg, rs=rs)
    print(f'{name}: robust-support classes = {[classes[i] for i in rs]}')

## One run in detail (see the training and a single 2x2)

In [ ]:
# ONE illustrative run (watch the CNN train, then the per-model table)
def one_run(name, seed=42, quiet_train=False):
    d = data[name]; Xtr, ytr, Xte, yte, classes, cfg, rs = (d['Xtr'], d['ytr'], d['Xte'], d['yte'], d['classes'], d['cfg'], d['rs'])
    cw = None
    if cfg.get('cnn_class_weights'):
        w = compute_class_weight('balanced', classes=np.unique(ytr), y=ytr)
        cw = torch.tensor(w, dtype=torch.float32, device=DEVICE)
    ctx = contextlib.redirect_stdout(io.StringIO()) if quiet_train else contextlib.nullcontext()
    with ctx:
        base = train_cnn(CNN1D(Xtr.shape[1], len(classes)), Xtr, ytr, n_epochs=50, device=DEVICE, class_weights=cw, random_seed=seed)
        defended = adversarial_train_cnn(Xtr, ytr, strategy='pgd', n_epochs=50, device=DEVICE, class_weights=cw, random_seed=seed)
    rf = build_random_forest(random_seed=seed).fit(Xtr, ytr)
    bcf = atk.wrap_cnn_for_art(base, Xtr.shape[1], len(classes), DEVICE)
    dcf = atk.wrap_cnn_for_art(defended, Xtr.shape[1], len(classes), DEVICE)
    Xt = atk.generate_pgd(bcf, Xte, 0.10)   # transfer (from baseline)
    Xw = atk.generate_pgd(dcf, Xte, 0.10)   # white-box (from defended)
    def f(clf, X): return mf1(yte, clf.predict(X).argmax(1), rs)
    return dict(base=(f(bcf, Xte), f(bcf, Xt), f(bcf, Xt)),
                defended=(f(dcf, Xte), f(dcf, Xt), f(dcf, Xw)),
                rf=(mf1(yte, rf.predict(Xte), rs), mf1(yte, rf.predict(Xt), rs), None))

for name in DATASETS:
    print('########## ' + name + ' ##########')
    r = one_run(name, seed=42, quiet_train=False)
    print(name + ': robust-support macro-F1, PGD eps=0.10')
    print(f"  {'model':15s}{'clean':>10}{'transfer':>10}{'white-box':>10}")
    for lab, key in [('baseline CNN','base'), ('defended CNN','defended'), ('Random Forest','rf')]:
        c, t, w = r[key]
        ws = 'n/a' if w is None else f'{w:.3f}'
        print(f"  {lab:15s}{c:>10.3f}{t:>10.3f}{ws:>10}")

## The decisive result - averaged over repeats (mean +/- std)

In [ ]:
# AVERAGED decisive result: repeat the whole thing and report mean +/- std.
# CICIoV single runs are unstable (CUDA nondeterminism on tiny data), so only the average is trustworthy.
N_REPEATS = 3
keys = ['base_clean','base_atk','def_clean','def_transfer','def_whitebox','rf_clean','rf_atk']
agg = {name: {k: [] for k in keys} for name in DATASETS}
for r in range(N_REPEATS):
    for name in DATASETS:
        res = one_run(name, seed=42 + r, quiet_train=True)
        A = agg[name]
        A['base_clean'].append(res['base'][0]); A['base_atk'].append(res['base'][1])
        A['def_clean'].append(res['defended'][0]); A['def_transfer'].append(res['defended'][1]); A['def_whitebox'].append(res['defended'][2])
        A['rf_clean'].append(res['rf'][0]); A['rf_atk'].append(res['rf'][1])
    print(f'repeat {r + 1}/{N_REPEATS} done')

def ms(v): return f'{np.mean(v):.3f}+/-{np.std(v):.3f}'
print()
print('robust-support macro-F1, mean +/- std over %d repeats, PGD eps=0.10' % N_REPEATS)
print(f"  {'dataset':12s}{'model':15s}{'clean':>16}{'transfer':>16}{'white-box':>16}")
for name in DATASETS:
    A = agg[name]
    print(f"  {name:12s}{'baseline CNN':15s}{ms(A['base_clean']):>16}{ms(A['base_atk']):>16}{ms(A['base_atk']):>16}")
    print(f"  {'':12s}{'defended CNN':15s}{ms(A['def_clean']):>16}{ms(A['def_transfer']):>16}{ms(A['def_whitebox']):>16}")
    print(f"  {'':12s}{'Random Forest':15s}{ms(A['rf_clean']):>16}{ms(A['rf_atk']):>16}{'n/a':>16}")